In [100]:
import pandas as pd
import tensorflow as tf
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.layers import Layer
import mediapipe as mp
from mrcnn.config import Config
from mrcnn import model as modellib
from mrcnn import visualize
import cv2
import numpy as np
import random
from matplotlib import pyplot
from PIL import Image

#for a cleaner output
import warnings
warnings.filterwarnings("ignore")

In [101]:
#get path to annotation file, contains coordinates of where the clothing item is in each image
bbox_file = "C:/Users/mikae/Downloads/In-shop Clothes Retrieval Benchmark-004/In-shop Clothes Retrieval Benchmark/Anno/list_bbox_inshop.txt"

# Load dataset using pandas
df = pd.read_csv(bbox_file, delim_whitespace=True, skiprows=2, header=None)

#renaming columns for simplicity
df.columns = ['image_path', 'class_id', 'image_id', 'x1', 'y1', 'x2', 'y2']
df['class_id'] = df['image_path'].apply(lambda x: x.split('/')[2])  # e.g., "Blouses_Shirts", "Dresses"
df['class_id'] = df['class_id'].str.lower()

#display first rows
print(df.head())

# Get column names
print(df.columns)

                                          image_path        class_id  \
0  img/WOMEN/Blouses_Shirts/id_00000001/02_1_fron...  blouses_shirts   
1  img/WOMEN/Blouses_Shirts/id_00000001/02_2_side...  blouses_shirts   
2  img/WOMEN/Blouses_Shirts/id_00000001/02_3_back...  blouses_shirts   
3  img/WOMEN/Blouses_Shirts/id_00000001/02_4_full...  blouses_shirts   
4       img/WOMEN/Dresses/id_00000002/02_1_front.jpg         dresses   

   image_id   x1  y1   x2   y2  
0         1   50  49  208  235  
1         2  119  48  136  234  
2         3   50  42  213  240  
3         4   82  30  162  129  
4         1   65  45  233  252  
Index(['image_path', 'class_id', 'image_id', 'x1', 'y1', 'x2', 'y2'], dtype='object')


In [102]:
#create the full image path
base_dir = r"C:\Users\mikae\Downloads\In-shop Clothes Retrieval Benchmark-004\In-shop Clothes Retrieval Benchmark\Anno\densepose\img_iuv"
#change image path column to full path
df['full_path'] = df['image_path'].apply(lambda x: os.path.join(base_dir, x))

In [103]:
#Functions for processing image
def estimate_body_shape(row):
    width = row['x2'] - row['x1']
    height = row['y2'] - row['y1']
    aspect_ratio = width / height
    if aspect_ratio > 0.45:
        return 'curvy'
    elif aspect_ratio < 0.35:
        return 'slim'
    else:
        return 'average'

#loads and prepares each image
def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path) #load image from file
    image = tf.image.decode_jpeg(image, channels=3) #decode jpeg
    image = tf.image.resize(image, [224, 224]) #resize to standard input size
    image = image / 255.0  #normalization
    return image, label #returns the image, label pair

#used for user input
def preprocess_user_image(image_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [224, 224])  # Match the DeepFashion2 image size
    image = image / 255.0  # Normalize pixel values to 0-1
    return image

In [104]:
# Convert columns to numpy arrays
image_paths = df['full_path'].values 
labels = df['class_id'].values 

#create tensorflow dataset using paths and labels, this is a paired dataset matching image_paths to labels 
dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
dataset = dataset.map(load_and_preprocess_image)

#shuffle the batch to sample data
dataset = dataset.shuffle(buffer_size=1000).batch(32)

'''
#sample some data and visualize it
for images, labels in dataset.take(1):
    plt.figure(figsize=(12,6))
    for i in range(8):
        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(images[i])
        plt.title(f'Label: {labels[i].numpy()}')
        plt.axis("off")
'''

'\n#sample some data and visualize it\nfor images, labels in dataset.take(1):\n    plt.figure(figsize=(12,6))\n    for i in range(8):\n        ax = plt.subplot(2, 4, i + 1)\n        plt.imshow(images[i])\n        plt.title(f\'Label: {labels[i].numpy()}\')\n        plt.axis("off")\n'

In [105]:
#config for mask r cnn
class InferenceConfig(Config):
    NAME = "coco"
    NUM_CLASSES = 81  #80 object categories and one background class
    GPU_COUNT = 1 #controls batch size
    IMAGES_PER_GPU = 1 #controls batch size
    IMAGE_MIN_DIM = 800
    IMAGE_MAX_DIM = 1024 #control image resizing
    RPN_ANCHOR_SCALES = (32, 64, 128, 256, 512) #allows for detection of various sizes



In [106]:
#create instance of config
config = InferenceConfig()

#sets mode to inference (prediction), sets directory where files and logs are saved, applies config settings to model
model = modellib.MaskRCNN(mode="inference", model_dir='./', config=config)

#apply weights for neural network
#coco has pretrained weights
model.load_weights('mask_rcnn_coco.h5', by_name=True)


In [107]:
clothing_aliases = {
    "dress": "dresses",
    "dresses": "dresses",
    "hoodie": "sweatshirts_hoodies",
    "sweatshirt": "sweatshirts_hoodies",
    "hoodies": "sweatshirts_hoodies",
    "graphic tee": "graphic_tees",
    "graphic t-shirt": "graphic_tees",
    "tank top": "tees_tanks",
    "tank": "tees_tanks",
    "tee": "tees_tanks",
    "t-shirt": "tees_tanks",
    "blouse": "blouses_shirts",
    "shirt": "blouses_shirts",
    "sweater": "sweaters",
    "cardigan": "cardigans",
    "jacket": "jackets_coats",
    "coat": "jackets_coats",
    "jumpsuit": "rompers_jumpsuits",
    "romper": "rompers_jumpsuits",
    "shorts": "shorts",
    "pants": "pants",
    "trousers": "pants",
    "skirt": "skirts",
    "leggings": "leggings",
    "vest": "jackets_vests",
    "jacket": "jackets_vests",
    "suit": "suiting",
    "polo": "shirts_polos"
}



In [108]:
#get candidate recommendations for user preference
#input will likely look something like this:
user_pref = {
  "body_shape": "curvy",
  "clothing_type": "dress",
  "posture": "standing",
    "gender": "female"
}

#add body shape estimates
df['aspect_ratio'] = (df['x2'] - df['x1']) / (df['y2'] - df['y1']).replace(0, 1)
df['body_shape'] = df.apply(estimate_body_shape, axis=1)

#normalize user input to match a class_id
user_input_clothing = user_pref["clothing_type"].strip()
standard_class = clothing_aliases.get(user_input_clothing)
if standard_class:
    user_pref["clothing_type"] = standard_class
else:
    raise ValueError(f"Unrecognized clothing type: {user_input_clothing}")

if user_pref['gender'].lower() == "female":
    gender_folder = "WOMEN"
else:
    gender_folder = "MEN"

#get possible options for user
candidates_df = df[
    (df['class_id'] == user_pref['clothing_type'].lower()) &
    (df['body_shape'] == user_pref['body_shape']) &
    (df['image_path'].str.contains(gender_folder, case=False))
]

'''
print(f"Number of candidate images found: {len(candidates_df)}")
print(candidates_df[['image_path']].head())

candidate_image_path = os.path.join(base_dir, candidates_df.iloc[0]['image_path'])  # Get the path of the first matching image
candidate_image = Image.open(candidate_image_path)  # Open the image
plt.imshow(candidate_image)
plt.axis('off')
plt.show()
print(candidate_image)

print("Recommended Image Path:", candidate_image_path)
'''



'\nprint(f"Number of candidate images found: {len(candidates_df)}")\nprint(candidates_df[[\'image_path\']].head())\n\ncandidate_image_path = os.path.join(base_dir, candidates_df.iloc[0][\'image_path\'])  # Get the path of the first matching image\ncandidate_image = Image.open(candidate_image_path)  # Open the image\nplt.imshow(candidate_image)\nplt.axis(\'off\')\nplt.show()\nprint(candidate_image)\n\nprint("Recommended Image Path:", candidate_image_path)\n'

In [109]:
#use mediapipe and opencv to get keypoints from the user and a batch of candidate images
mp_pose = mp.solutions.pose
pose_model = mp_pose.Pose(static_image_mode=True)

def extract_keypoints(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w, _ = image.shape
    if max(h, w) > 512:
        scale = 512 / max(h, w)
        image = cv2.resize(image, (int(w * scale), int(h * scale)))

    results = pose_model.process(image)

    if results.pose_landmarks:
        keypoints = [(lm.x, lm.y) for lm in results.pose_landmarks.landmark]
        return keypoints
    else:
        return None  # No person detected

def pose_similarity(user_keypoints, candidate_keypoints, important_indices):
    distances = []
    for idx in important_indices:
        if idx < len(user_keypoints) and idx < len(candidate_keypoints):
            user_pt = user_keypoints[idx]
            cand_pt = candidate_keypoints[idx]
            if user_pt and cand_pt:
                dist = np.linalg.norm(np.array(user_pt) - np.array(cand_pt))
                distances.append(dist)
    if distances:
        return np.mean(distances)
    else:
        return float('inf')


In [112]:
# Load user image and extract keypoints
user_image_path = "C:/Users/mikae/OneDrive/Pictures/IMG_4232.JPG"  # hardcoded for now
user_keypoints = extract_keypoints(user_image_path)

if user_keypoints is None:
    raise ValueError("No pose detected in user image.")

#get important keypoints for comparison
important_points = [11, 12, 23, 24]  
best_distance = float('inf')
best_candidate_path = None

#limit candidates to 100 by randomly selecting images
candidate_paths = candidates_df['image_path'].tolist()
random.shuffle(candidate_paths)
candidate_paths = candidate_paths[:100] 

#go through each candidate path and evaluate
for idx, relative_path in enumerate(candidate_paths):
    full_path = os.path.join(base_dir, relative_path)
    candidate_keypoints = extract_keypoints(full_path)

    if candidate_keypoints is None:
        print(f"Skipping {relative_path}: No keypoints detected.")
        continue

    distance = pose_similarity(user_keypoints, candidate_keypoints, important_points)
    print(f"[{idx+1}] Checked {relative_path} | Pose Distance: {distance:.2f}")

    if distance < best_distance:
        best_distance = distance
        best_candidate_path = full_path

# Final check
if best_candidate_path is None:
    raise ValueError("No match found.")
else:
    print(f'Best match: {best_candidate_path}')
    print(f'Best pose similarity distance: {best_distance:.2f}')
    best_image = Image.open(best_candidate_path)
    plt.imshow(best_image)
    plt.axis('off')
    plt.title("Best Matching Candidate")
    plt.show()


error: OpenCV(4.11.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'
